## Loading libraries and data

Here we load the necessary libraries. Note that while `pandas`, `numpy`, `sklearn`, `statsmodels` come with Anaconda distribution, you will probably have to install `stargazer` (for printing regression tables) and `plotly` (data visualization) manually.

In [1]:
#@title
# This is to install some packages
!pip install stargazer
!pip install plotly
# You can comment it out later

In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.model_selection import train_test_split

import statsmodels.api as sm
import statsmodels.formula.api as smf

from stargazer.stargazer import Stargazer, LineLocation
import plotly.express as px

# Notebook 2: logistic regression

Go through the notebook and do exercises. There will be topics for discussion in the end - we will discuss them in the next class.

By the end of this activity, you should

1. Learn how to train and evaluate a logistic regression model

2. Explain how to recognize overfitting

3. Explain importance of variable selection

The following command imports the CSV dataset using pandas:

In [ ]:
#### Loading data from Google Drive - thanks to ChatGPT
# https://drive.google.com/file/d/1krC17OblVQ9tsEshKmv2hGBAeqJlqn83/view?usp=sharing

import requests
from io import StringIO

# Set the file ID of the CSV file you want to load
file_id = "1krC17OblVQ9tsEshKmv2hGBAeqJlqn83"

# Set the URL to download the file using the Drive API
url = f"https://drive.google.com/uc?id={file_id}&export=download"

# Make a GET request to download the file and decode the content
content = requests.get(url).content.decode("utf-8")

# Convert the string content to a pandas dataframe
dataset = pd.read_csv(StringIO(content))
dataset.shape

Here is what the data look like. Note that "d_k" here is the bitcoin price $k$ days ago for $k=0,1,2,\dots,500$.

In [ ]:
dataset.head()

And some simple description

In [ ]:
dataset.describe()

## Data exploration

The variable "d_0" is the price of bitcoin at a given day. Below is the time series plot.

In [ ]:
fig = px.line(dataset,  y = 'd_0')
fig.update_layout(xaxis_title='Day', yaxis_title = "Bitcoin price")
fig.show()

## Data processing

This is an exercise on classification, i.e., we will be interested in whether the price of bitcoin goes up or down. We will add a new variable called "move". Its value will be 1 if the bitcoin price goes up and 0 if it goes down or stays the same.

In [ ]:
dataset["move"] = 0.0 + (dataset['d_0'] > dataset['d_1'])
dataset

## Training and test datasets

Here we will split the entire dataset into 70% training and 30% test data. We will also remove the original variable "d_0" (think why).

In [ ]:
train_data, test_data = train_test_split(dataset, test_size=0.3, random_state = 8128)
train_data = train_data.drop('d_0', axis = 1)
test_data = test_data.drop('d_0', axis = 1)
print("Entire dataset dimensions =", dataset.shape)
print("Train data dimensions =", train_data.shape)
print("Test data dimensions =", test_data.shape)

### Exercise 1

Does the bitcoin price more often go up or down in the training data? Imagine that instead of training a logistic regression, we will use a super-simple baseline model - always predict that the price goes up or down (whichever is more common in the training data). What will be the test accuracy of such a baseline model?

In [ ]:
### Write your code here

fraction_up_days = np.mean(train_data['move'])
print("Fraction of days on which the price goes up in the training data is", np.round(fraction_up_days, 2))
baseline_prediction = 0. + (fraction_up_days > 0.5)
print("The baseline prediction is", baseline_prediction)
baseline_test_acc = np.mean(0. + (test_data['move'] == baseline_prediction))
print("Test accuracy of baseline model =", baseline_test_acc)

## Modelling

Here we train two logistic models. The first one uses 5 historical prices as predictors and the second one uses 300 historical prices as predictors. For the second regression, we will create a formula in a loop (let Fedor know if there is a better method) as follows:

In [ ]:
f = "move ~" + " d_1"
for i in range(2, 301):
    f = f + " + d_" + str(i)

print(f[0:100])

In [ ]:
mod1 = smf.logit(formula = 'move ~ d_1 + d_2 + d_3 + d_4 + d_5', data = train_data).fit()
mod2 = smf.logit(formula = f, data = train_data).fit()

stargazer = Stargazer([mod1])
# We will not report coefficients of the second model since there are too many of them
stargazer

### Model evaluation

To evaluate our model, we need some performance metric. Such a performance metric can either show how good a model is (accuracy) or how bad a model is (error or loss function). For classification models, we usually use metrics related to accuracy (accuracy, balanced accuracy, precision, recall, F1-score etc). Here we will just look at the overall accuracy, i.e.,
$$
\frac{\mbox{Number of correctly predicted observations}}{\mbox{Total number of observations}}
$$

Note that the accuracy is not a differentiable function and hence its impossible to use it to train logistic regression by gradient descent. To train logistic regression, we use the binary cross-entropy loss
$$
L(\beta)=-\sum_{i=1}^{N}\left(y^i\log p(x^i)+ (1-y^i)\log(1-p(x^i))\right)
$$
It means that, by definition, model parameters are found as to minimize it, i.e.,
$$
(\beta_0,\beta_1,\dots,\beta_p)=\arg\min L
$$
However, the binary cross-entropy loss is not interpretable and it doesn't make much sense to use it to evaluate the final model.

First, we will construct predictions of model 1 (with 5 predictors).

In [ ]:
mod1.predict(test_data)

Note that logistic regression reports probabilities rather than labels. This is actually a good thing because not only it tells us which label it predicts, it also report how confident it is in its predictions. We will just use the standard convention that if the probability is above 0.5, then the predicted label is 1, otherwise it is 0. Here are some predicted labels.

In [ ]:
0. + (mod1.predict(test_data) > 0.5)

Now we are ready to construct predictions and report training and test accuracies of the two models

In [ ]:
pred1_train = 0. + (mod1.predict(train_data) > 0.5)
pred1_test = 0. + (mod1.predict(test_data) > 0.5)

# Note that since we trained the 500-regressor model in a different manner (without the formula API),
# we need to use the predict method in a different way too:
pred2_train = 0. + (mod2.predict(train_data) > 0.5)
pred2_test = 0. + (mod2.predict(test_data) > 0.5)

print("Training accuracy of model 1 =", metrics.accuracy_score(train_data["move"], pred1_train))
print("Test accuracy of model 1 =", metrics.accuracy_score(test_data["move"], pred1_test))
print("Training accuracy of model 2 =", metrics.accuracy_score(train_data["move"], pred2_train))
print("Test accuracy of model 2 =", metrics.accuracy_score(test_data["move"], pred2_test))

### Exercise 2

Write a Python function with three inputs - a logistic regression model, a dataset, and the name of the column containing the response variable. The output of such a function should be the overall accuracy of the model. You will need to modify the following code from Notebook 1 to do that:

In [ ]:
# Modify this code
# Change the name of the function "model_mae" to "model_acc" and do other necessary changes

def model_acc(model, data = test_data, y = 'move'):
    pred = 0. + (model.predict(data) > 0.5)
    return metrics.accuracy_score(data[y], pred)

print("Test accuracy of model 1 =", model_acc(mod1))
print("Test accuracy of model 2 =", model_acc(mod2))

## Exercise 3

Look at the baseline model again. Is the logistic regression better than the baseline model? What if we use fewer predictors or more predictors?

In [ ]:
print("Test accuracy of model 1 =", metrics.accuracy_score(test_data["move"], pred1_test))
print("Test accuracy of baseline model =", baseline_test_acc)

Test accuracy of the baseline model appears to be just a little bit lower than the test accuracy of the first logistic regression. The difference is, however, so small, that the situation may be reversed if we had a different split into training and test sets.

We will try to fit a univariate logistic regression to check if it makes more sense to go with fewer regressors

In [ ]:
mod3 = smf.logit(formula = 'move ~ d_1', data = train_data).fit()
print("Test accuracy of model 3 =", model_acc(mod3))

It seems to make more sense to use just one regressor here.

## Question to think about

Think about these questions at home - we will discuss them in next class

### Question 1 - super-complex model

Why do you think does the second model appear to be so much more accurate on the training data but performs so poorly on the test data? What do we do to prevent such a situation?

**FEDOR'S EXPLANATION** The second model overfits because it is too flexible. To prevent overfitting, we need to do regularization or feature selection.

### Question 2 - variable selection

Our experiments show that we can try models with a different number of predictors and it is important to choose the number of predictors wisely. What do you think of the following way to do it?

We will train logistic regressions $f(X_1), f(X_1,X_2), f(X_1,X_2,X_3),\ldots, f(X_1,X_2,\dots,X_{500})$ (where $X_d$ is the bitcoin price $d$ days ago). Every one of them will be trained on the train data and but we will use the test accuracy to evaluate its performance. Then we will choose the model with the highest test accuracy.

Is it a correct way to choose the right number of predictors?

**FEDOR'S EXPLANATION** This is an incorrect method because choosing variables that are included into the model is a part of training and we cannot use the test set for training.

# Theoretical homework

### Problem 1

A linear regression is usually trained by minimizing the mean squared error loss
$$
L(\beta)=\frac{1}{N}\sum_{i=1}^{N}\left(y^i-\beta_0-
\sum_{j=1}^{p}\beta_j x^i_j\right)^2
$$
The reason is that the first order conditions $\frac{\partial L}{\partial \beta_i}=0$ for $i=0,1,\dots,p$ can be solved in closed form (you learned how to do it in regression analysis). However, it is easier to interpret the mean absolute error, i.e.,
$$
\frac{1}{N}\sum_{i=1}^{N}\left|y^i-\beta_0-
\sum_{j=1}^{p}\beta_j x^i_j\right|
$$
Compute partial derivatives of the mean absolute error with respect to model parameters. Is the mean absolute error continuous? differentiable? If not, is non-continuity or non-differentiability going to be a problem for training the model by gradient descent?

### Answer

Here, we need to use the following simple result:
$$
\frac{d}{dx}|x|=
\begin{cases}
1, & x > 0, \\
\mbox{undefined}, & x=0 \\
-1, & x<0
\end{cases}=\frac{|x|}{x}
$$

Differentiating the mean absolute error loss
$$
L(\beta)=\frac{1}{N}\sum_{i=1}^{N}\left|y^i-\beta_0-
\sum_{j=1}^{p}\beta_j x^i_j\right|
$$
with respect to $\beta_0$, we get
$$
\frac{\partial L}{\partial \beta_0}=
-\frac{1}{N}\sum_{i=1}^{N}\frac{\left|y^i-\beta_0-
\sum_{j=1}^{p}\beta_j x^i_j\right|}{y^i-\beta_0-
\sum_{j=1}^{p}\beta_j x^i_j}
$$
with respect to $\beta_j$, $j>0$, we get
$$
\frac{\partial L}{\partial \beta_j}=
-\frac{1}{N}\sum_{i=1}^{N}\frac{\left|y^i-\beta_0-
\sum_{j=1}^{p}\beta_j x^i_j\right|}{y^i-\beta_0-
\sum_{j=1}^{p}\beta_j x^i_j}\times x^i_j
$$
The mean absolute error loss is continuous, but it is not differentiable at some points. Specifically, it is not differentiable for parameter values such that the model's prediction on some training data point is exactly equal to the observed value. These parameter values are exceptional. Most likely, this is not going to be a problem for gradient descent.

### Problem 2

Recall from the lecture that the loss function for logistic regression is the binary cross-entropy
$$
L(\beta)=-\sum_{i=1}^{N}\left(y^i\log p(x^i)+ (1-y^i)\log(1-p(x^i))\right)
$$
Compute its gradient, i.e., find partial derivatives
$$
\frac{\partial L}{\partial \beta_i}
$$
for $i=0,1,\dots,p$.

### Answer:

First, we will find the derivative of the sigmoid function:
$$
\frac{ds}{dz}=
\frac{d}{dz}\frac{1}{1+e^{-z}}
=
-\frac{-e^{-z}}{(1+e^{-z})^2}=e^{-z}\cdot s^2(z)
$$
Another useful expression is that for $1-s(z)$. We have
$$
1-s(z)=1-\frac{1}{1+e^{-z}}=
\frac{e^{-z}}{1+e^{-z}}=e^{-z}\cdot s(z)
$$
Now we have
\begin{align*}
\frac{\partial}{\partial \beta^{T}}L(\beta)=&
\frac{\partial}{\partial \beta^{T}}
\left(
-\sum_{i=1}^{N}y^i\log s\left(x^{iT}\beta\right) -
\sum_{i=1}^{N}(1-y^i)\log\left(1- s\left(x^{iT}\beta\right)\right)
\right)\\
=&
-\sum_{i=1}^{N}y^i
\frac{1}{s\left(x^{iT}\beta\right)}\cdot
e^{-X^T\beta}\cdot s^2\left(x^{iT}\beta\right)\cdot
\frac{\partial}{\partial \beta^{T}} \left(x^{iT}\beta\right)\\
&-\sum_{i=1}^{N}(1-y^i)\cdot \frac{1}{1-s\left(x^{iT}\beta\right)}
\cdot -e^{-x^{iT}\beta}\cdot s^2\left(x^{iT}\beta\right)
\frac{\partial}{\partial \beta^{T}} \left(x^{iT}\beta\right)
\end{align*}
Further, notice that
$$
\frac{\partial}{\partial \beta^{T}} \left(x^{iT}\beta\right)=
\left(\frac{\partial}{\partial \beta_0},
\frac{\partial}{\partial \beta_1},
\cdots,
\frac{\partial}{\partial \beta_p}
\right)\left(\beta_0, x^i_1\beta_1,\cdots,
x^i_p\beta_p\right)=
\left(1,x^i_1,\cdots,x^i_p\right)=x^{iT}
$$
Applying the properties of the sigmoid function, we get
$$
\frac{1}{s\left(x^{iT}\beta\right)}\cdot
e^{-X^T\beta}\cdot s^2\left(x^{iT}\beta\right)=
e^{-X^T\beta}\cdot s\left(x^{iT}\beta\right)=
1-s\left(x^{iT}\beta\right)=1-\hat{y}^i
$$
and, similarly,
$$
\frac{1}{1-s\left(x^{iT}\beta\right)}
\cdot -e^{-x^{iT}\beta}\cdot s^2\left(x^{iT}\beta\right)=
e^{x^{iT}\beta}\cdot
\frac{1}{s\left(x^{iT}\beta\right)}
\cdot -e^{-x^{iT}\beta}\cdot s^2\left(x^{iT}\beta\right)
=-\hat{y}^i
$$
Putting it all together yields
$$
\frac{\partial}{\partial \beta^{T}}L(\beta)=
-\sum_{i=1}^{N}y^i(1-\hat{y}^i)x^{iT}+
\sum_{i=1}^{N}(1-y^i)\hat{y}^ix^{iT}
=-\sum_{i=1}^{N}(y^i-\hat{y}^i)x^{iT}
=-\mathbf{X}^{T}(\mathbf{y}-\mathbf{\hat{y}})
$$

### Problem 3

Explain why adding more predictors to a linear model will always reduce the training mean squared error. Is it true that adding more predictors to the logistic model will always increase the training accuracy? Either prove it or provide a counter-example.

### Answer

A linear model is trained by minimizing the mean squared error
$$
L(\beta)=\frac{1}{N}\sum_{i=1}^{N}\left(y^i-\beta_0-
\sum_{j=1}^{p}\beta_j x^i_j\right)^2
$$
It means that the estimated parameter values are obtained via
$$
\hat{\beta}=\arg\min L(\beta)
$$
and the training error is
$$
\min L(\beta)=L(\hat{\beta})
$$

Let's say now we include more parameters to the model, i.e., consider a new model with parameters $(\beta,\gamma)$ trained by minimizing the mean squared error
$$
\tilde{L}(\beta,\gamma)
$$
Here, $\gamma$ is the vector of the extra parameters. Then the new training error will be
$$
\min \tilde{L}(\beta,\gamma)
$$
and we need to show that
$$
\min \tilde{L}(\beta,\gamma)\le \min L(\beta)
$$
This trivially follows from the fact that $\tilde{L}(\beta,0)= L(\beta)$, i.e., the range of $L$ is a subset in the range of $\tilde{L}$.

According to the same logic, the training loss of the logistic regression will always decrease if we include more parameters into the model. However, the accuracy of logistic regression may not. It's not even true that the logistic regression parameters that minimize the training loss will also always maximize the training accuracy.

Fedor doesn't know an example though.